<a href="https://colab.research.google.com/github/sabinrajpokhrel/Wildfire_ML_Project/blob/main/ERA5_DATA_PIPELINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade earthengine-api geemap
!pip install -q \
    google-cloud-bigquery \
    google-cloud-bigquery-storage \
    db-dtypes \
    pyarrow

In [1]:
# Authenticating the Earth Engine to the project
import ee
import geemap

ee.Authenticate()

ee.Initialize(
    project="wildfire-project-502905"
)

print("Earth Engine initialized successfully.")

Earth Engine initialized successfully.


In [2]:
# Loading Nepal's Geographical Boundary

countries = ee.FeatureCollection(
    "FAO/GAUL/2015/level0"
)

nepal = countries.filter(
    ee.Filter.eq("ADM0_NAME", "Nepal")
)

nepal_geometry = nepal.geometry()

print("Number of Nepal boundary features:", nepal.size().getInfo())

Number of Nepal boundary features: 1


In [3]:
# Displaying Nepal on Map
Map = geemap.Map(
    center=[28.3, 84.1],
    zoom=6
)

Map.addLayer(
    nepal.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Nepal boundary"
)

Map

Map(center=[28.3, 84.1], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [4]:
# Checking Nepal's Bounding rectangle
nepal_bounds = nepal_geometry.bounds()

print(
    nepal_bounds.getInfo()
)

{'geodesic': False, 'type': 'Polygon', 'coordinates': [[[80.05875381298702, 26.353577038140237], [88.20193296831302, 26.353577038140237], [88.20193296831302, 30.44968051484387], [80.05875381298702, 30.44968051484387], [80.05875381298702, 26.353577038140237]]]}


In [20]:
# Complete modelling period
ERA5_START_DATE = "2025-01-05"

# Earth Engine end dates are exclusive.
# This includes data through 19 May 2026.
ERA5_END_DATE_EXCLUSIVE = "2026-05-20"

In [7]:
ERA5_BANDS = [
    "temperature_2m",
    "temperature_2m_max",
    "dewpoint_temperature_2m",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "total_precipitation_sum",
    "volumetric_soil_water_layer_1"
]

In [8]:
era5_raw = (
    ee.ImageCollection(
        "ECMWF/ERA5_LAND/DAILY_AGGR"
    )
    .filterDate(
        ERA5_START_DATE,
        ERA5_END_DATE_EXCLUSIVE
    )
    .filterBounds(
        nepal_geometry
    )
    .select(
        ERA5_BANDS
    )
)

print(
    "Number of daily ERA5 images:",
    era5_raw.size().getInfo()
)

Number of daily ERA5 images: 500


In [9]:
first_era5_image = ee.Image(
    era5_raw.first()
)

print(
    first_era5_image.bandNames().getInfo()
)

['temperature_2m', 'temperature_2m_max', 'dewpoint_temperature_2m', 'u_component_of_wind_10m', 'v_component_of_wind_10m', 'total_precipitation_sum', 'volumetric_soil_water_layer_1']


In [10]:
first_date = ee.Date(
    first_era5_image.get(
        "system:time_start"
    )
).format("YYYY-MM-dd")

print(
    "First ERA5 date:",
    first_date.getInfo()
)

First ERA5 date: 2025-01-05


In [11]:
def prepare_era5_image(image):
    image = ee.Image(image)

    # Daily mean temperature: Kelvin to Celsius
    temperature_c = (
        image
        .select("temperature_2m")
        .subtract(273.15)
        .rename("temperature_mean_c")
    )

    # Daily maximum temperature: Kelvin to Celsius
    temperature_max_c = (
        image
        .select("temperature_2m_max")
        .subtract(273.15)
        .rename("temperature_max_c")
    )

    # Dew point temperature: Kelvin to Celsius
    dewpoint_c = (
        image
        .select("dewpoint_temperature_2m")
        .subtract(273.15)
        .rename("dewpoint_c")
    )

    # Wind components
    u_wind = (
        image
        .select("u_component_of_wind_10m")
        .rename("u_wind_mps")
    )

    v_wind = (
        image
        .select("v_component_of_wind_10m")
        .rename("v_wind_mps")
    )

    # Wind speed = sqrt(u² + v²)
    wind_speed = (
        u_wind
        .pow(2)
        .add(v_wind.pow(2))
        .sqrt()
        .rename("wind_speed_mps")
    )

    # Precipitation: metres to millimetres
    precipitation_mm = (
        image
        .select("total_precipitation_sum")
        .max(ee.Image.constant(0))
        .multiply(1000)
        .rename("precipitation_mm")
    )

    # Surface soil moisture
    soil_moisture = (
        image
        .select("volumetric_soil_water_layer_1")
        .rename("soil_moisture")
    )

    # Relative humidity calculated from temperature and dew point
    relative_humidity = (
        ee.Image()
        .expression(
            expression="""
                100 *
                exp((17.625 * Td) / (243.04 + Td)) /
                exp((17.625 * T) / (243.04 + T))
            """,
            opt_map={
                "T": temperature_c,
                "Td": dewpoint_c
            }
        )
        .clamp(0, 100)
        .rename("relative_humidity")
    )

    prepared = ee.Image.cat([
        temperature_c,
        temperature_max_c,
        dewpoint_c,
        relative_humidity,
        u_wind,
        v_wind,
        wind_speed,
        precipitation_mm,
        soil_moisture
    ])

    return prepared.copyProperties(
        image,
        ["system:time_start"]
    )

In [12]:
era5_prepared = era5_raw.map(prepare_era5_image)

first_prepared_image = ee.Image(era5_prepared.first())

print(first_prepared_image.bandNames().getInfo())

['temperature_mean_c', 'temperature_max_c', 'dewpoint_c', 'relative_humidity', 'u_wind_mps', 'v_wind_mps', 'wind_speed_mps', 'precipitation_mm', 'soil_moisture']


In [13]:
# Period 1:
# Inclusive: 2025-01-05 to 2026-01-05
period_1 = era5_prepared.filterDate(
    "2025-01-05",
    "2026-01-06"   # Exclusive end date
)

# Period 2:
# Inclusive: 2026-01-06 to 2026-05-19
period_2 = era5_prepared.filterDate(
    "2026-01-06",
    "2026-05-20"   # Exclusive end date
)

print("Period 1 daily images:", period_1.size().getInfo())
print("Period 2 daily images:", period_2.size().getInfo())

Period 1 daily images: 366
Period 2 daily images: 134


In [14]:
def show_collection_dates(collection, name):
    collection = ee.ImageCollection(collection)

    first_image = ee.Image(
        collection.sort("system:time_start").first()
    )

    last_image = ee.Image(
        collection.sort(
            "system:time_start",
            False
        ).first()
    )

    first_date = ee.Date(
        first_image.get("system:time_start")
    ).format("YYYY-MM-dd")

    last_date = ee.Date(
        last_image.get("system:time_start")
    ).format("YYYY-MM-dd")

    print(
        name,
        "| images:", collection.size().getInfo(),
        "| first:", first_date.getInfo(),
        "| last:", last_date.getInfo()
    )

In [15]:
show_collection_dates(
    period_1,
    "Period 1"
)

show_collection_dates(
    period_2,
    "Period 2"
)

Period 1 | images: 366 | first: 2025-01-05 | last: 2026-01-05
Period 2 | images: 134 | first: 2026-01-06 | last: 2026-05-19


In [16]:
GRID_POINTS_PER_DAY = 1361

period_1_estimated_rows = (
    period_1.size().getInfo()
    * GRID_POINTS_PER_DAY
)

period_2_estimated_rows = (
    period_2.size().getInfo()
    * GRID_POINTS_PER_DAY
)

print(
    "Estimated Period 1 rows:",
    period_1_estimated_rows
)

print(
    "Estimated Period 2 rows:",
    period_2_estimated_rows
)

Estimated Period 1 rows: 498126
Estimated Period 2 rows: 182374


In [17]:
def image_to_rows(image):
    image = ee.Image(image)

    image_date = ee.Date(
        image.get("system:time_start")
    ).format("YYYY-MM-dd")

    projection = (
        image
        .select("temperature_mean_c")
        .projection()
    )

    samples = image.sample(
        region=nepal_geometry,
        projection=projection,
        scale=11132,
        geometries=True,
        tileScale=4
    )

    def add_information(feature):
        feature = ee.Feature(feature)

        coordinates = (
            feature
            .geometry()
            .coordinates()
        )

        return feature.set({
            "date": image_date,
            "latitude": coordinates.get(1),
            "longitude": coordinates.get(0)
        })

    return samples.map(add_information)

def collection_to_rows(image_collection):
    image_collection = ee.ImageCollection(
        image_collection
    )

    image_list = image_collection.toList(
        image_collection.size()
    )

    nested_rows = image_list.map(
        image_to_rows
    )

    return ee.FeatureCollection(
        nested_rows
    ).flatten()

In [18]:
EXPORT_COLUMNS = [
    "date",
    "latitude",
    "longitude",
    "temperature_mean_c",
    "temperature_max_c",
    "dewpoint_c",
    "relative_humidity",
    "u_wind_mps",
    "v_wind_mps",
    "wind_speed_mps",
    "precipitation_mm",
    "soil_moisture"
]

In [21]:
def export_era5_collection(
    collection,
    description,
    filename
):
    rows = collection_to_rows(
        collection
    )

    task = ee.batch.Export.table.toDrive(
        collection=rows,
        description=description,
        folder="Wildfire_ERA5_Nepal",
        fileNamePrefix=filename,
        fileFormat="CSV",
        selectors=EXPORT_COLUMNS
    )

    task.start()

    print(
        "Export started:",
        description
    )

    print(
        task.status()
    )

    return task

In [22]:
task_period_1 = export_era5_collection(
    collection=period_1,
    description=(
        "ERA5_Nepal_Period_1_Full_Rerun_V2"
    ),
    filename=(
        "era5_nepal_period_1_full_rerun_v2"
    )
)

Export started: ERA5_Nepal_Period_1_Full_Rerun_V2
{'state': 'READY', 'description': 'ERA5_Nepal_Period_1_Full_Rerun_V2', 'priority': 100, 'creation_timestamp_ms': 1785052915172, 'update_timestamp_ms': 1785052915172, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'YTYUECZ4PYZPXRC2GRJN62B2', 'name': 'projects/wildfire-project-502905/operations/YTYUECZ4PYZPXRC2GRJN62B2'}


In [30]:
task_period_1.status()

{'state': 'COMPLETED',
 'description': 'ERA5_Nepal_Period_1_Full_Rerun_V2',
 'priority': 100,
 'creation_timestamp_ms': 1785052915172,
 'update_timestamp_ms': 1785052996172,
 'start_timestamp_ms': 1785052918195,
 'task_type': 'EXPORT_FEATURES',
 'destination_uris': ['https://drive.google.com/#folders/1qfHxnmUxi1bjYJylof0j36V_Kb9l-Vkr'],
 'attempt': 1,
 'batch_eecu_usage_seconds': 53.41313934326172,
 'id': 'YTYUECZ4PYZPXRC2GRJN62B2',
 'name': 'projects/wildfire-project-502905/operations/YTYUECZ4PYZPXRC2GRJN62B2'}

In [24]:
task_period_2 = export_era5_collection(
    collection=period_2,
    description=(
        "ERA5_Nepal_Period_2_Full_Rerun_V2"
    ),
    filename=(
        "era5_nepal_period_2_full_rerun_v2"
    )
)

Export started: ERA5_Nepal_Period_2_Full_Rerun_V2
{'state': 'READY', 'description': 'ERA5_Nepal_Period_2_Full_Rerun_V2', 'priority': 100, 'creation_timestamp_ms': 1785052940028, 'update_timestamp_ms': 1785052940028, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'WQAMXRO4G7DW4KHZ4PTUYI2U', 'name': 'projects/wildfire-project-502905/operations/WQAMXRO4G7DW4KHZ4PTUYI2U'}


In [31]:
task_period_2.status()

{'state': 'COMPLETED',
 'description': 'ERA5_Nepal_Period_2_Full_Rerun_V2',
 'priority': 100,
 'creation_timestamp_ms': 1785052940028,
 'update_timestamp_ms': 1785053029368,
 'start_timestamp_ms': 1785052944906,
 'task_type': 'EXPORT_FEATURES',
 'destination_uris': ['https://drive.google.com/#folders/1qfHxnmUxi1bjYJylof0j36V_Kb9l-Vkr'],
 'attempt': 1,
 'batch_eecu_usage_seconds': 22.848003387451172,
 'id': 'WQAMXRO4G7DW4KHZ4PTUYI2U',
 'name': 'projects/wildfire-project-502905/operations/WQAMXRO4G7DW4KHZ4PTUYI2U'}

In [37]:
from google.colab import auth

auth.authenticate_user()

print("Google Cloud authentication completed.")

Google Cloud authentication completed.


In [38]:
from pathlib import Path

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/Wildfire_ML_Project"
)

ERA5_EXPORT_FOLDER = Path(
    "/content/drive/MyDrive/Wildfire_ERA5_Nepal"
)

In [83]:
from google.colab import drive

drive.flush_and_unmount()

drive.mount(
    "/content/drive",
    force_remount=True
)

Mounted at /content/drive


In [84]:
from pathlib import Path

DRIVE_MOUNT = Path("/content/drive")

print("Mounted folders:")

for path in DRIVE_MOUNT.iterdir():
    print(path)

Mounted folders:
/content/drive/.shortcut-targets-by-id
/content/drive/MyDrive
/content/drive/.Trash-0
/content/drive/.Encrypted


In [85]:
from pathlib import Path

DRIVE_MOUNT = Path("/content/drive")

period_1_files = list(
    DRIVE_MOUNT.rglob(
        "era5_nepal_period_1_full_rerun_v2*.csv"
    )
)

period_2_files = list(
    DRIVE_MOUNT.rglob(
        "era5_nepal_period_2_full_rerun_v2*.csv"
    )
)

print("Period 1 matches:")

for path in period_1_files:
    print(path)

print("\nPeriod 2 matches:")

for path in period_2_files:
    print(path)

Period 1 matches:
/content/drive/MyDrive/Wildfire_ERA5_Nepal/era5_nepal_period_1_full_rerun_v2.csv

Period 2 matches:
/content/drive/MyDrive/Wildfire_ERA5_Nepal/era5_nepal_period_2_full_rerun_v2.csv


In [86]:
period_1_files = sorted(
    period_1_files
)

period_2_files = sorted(
    period_2_files
)

assert period_1_files, (
    "Period 1 file is not visible in the mounted Drive."
)

assert period_2_files, (
    "Period 2 file is not visible in the mounted Drive."
)

print(
    "Period 1 file:",
    period_1_files[0]
)

print(
    "Period 2 file:",
    period_2_files[0]
)

Period 1 file: /content/drive/MyDrive/Wildfire_ERA5_Nepal/era5_nepal_period_1_full_rerun_v2.csv
Period 2 file: /content/drive/MyDrive/Wildfire_ERA5_Nepal/era5_nepal_period_2_full_rerun_v2.csv


In [87]:
period_1_files = sorted(
    ERA5_EXPORT_FOLDER.glob(
        "era5_nepal_period_1_full_rerun_v2*.csv"
    )
)

period_2_files = sorted(
    ERA5_EXPORT_FOLDER.glob(
        "era5_nepal_period_2_full_rerun_v2*.csv"
    )
)

print("Period 1 files:")

for file_path in period_1_files:
    print(file_path.name)

print("\nPeriod 2 files:")

for file_path in period_2_files:
    print(file_path.name)

assert period_1_files, (
    "Period 1 export was not found."
)

assert period_2_files, (
    "Period 2 export was not found."
)

Period 1 files:
era5_nepal_period_1_full_rerun_v2.csv

Period 2 files:
era5_nepal_period_2_full_rerun_v2.csv


In [88]:
import pandas as pd


def load_csv_parts(file_paths):
    frames = []

    for file_path in file_paths:
        current_df = pd.read_csv(file_path)

        print(
            file_path.name,
            current_df.shape
        )

        frames.append(current_df)

    return pd.concat(
        frames,
        ignore_index=True
    )

In [89]:
era5_period_1 = load_csv_parts(
    period_1_files
)

era5_period_2 = load_csv_parts(
    period_2_files
)

print(
    "Period 1 shape:",
    era5_period_1.shape
)

print(
    "Period 2 shape:",
    era5_period_2.shape
)

era5_nepal_period_1_full_rerun_v2.csv (498126, 12)
era5_nepal_period_2_full_rerun_v2.csv (182374, 12)
Period 1 shape: (498126, 12)
Period 2 shape: (182374, 12)


In [90]:
era5 = pd.concat(
    [
        era5_period_1,
        era5_period_2
    ],
    ignore_index=True
)

In [91]:
era5["date"] = pd.to_datetime(
    era5["date"],
    errors="coerce"
).dt.normalize()

numeric_columns = [
    "latitude",
    "longitude",
    "temperature_mean_c",
    "temperature_max_c",
    "dewpoint_c",
    "relative_humidity",
    "u_wind_mps",
    "v_wind_mps",
    "wind_speed_mps",
    "precipitation_mm",
    "soil_moisture"
]

for column in numeric_columns:
    era5[column] = pd.to_numeric(
        era5[column],
        errors="coerce"
    )

era5 = (
    era5
    .sort_values(
        ["date", "latitude", "longitude"]
    )
    .reset_index(drop=True)
)

In [92]:
unique_grid_locations = (
    era5[
        ["latitude", "longitude"]
    ]
    .drop_duplicates()
    .shape[0]
)

duplicate_rows = era5.duplicated(
    [
        "date",
        "latitude",
        "longitude"
    ]
).sum()

missing_values = (
    era5.isna().sum().sum()
)

print("Dataset shape:", era5.shape)
print("First date:", era5["date"].min())
print("Last date:", era5["date"].max())
print("Unique dates:", era5["date"].nunique())
print("Unique grid locations:", unique_grid_locations)
print("Duplicate rows:", duplicate_rows)
print("Missing values:", missing_values)

assert era5.shape == (680_500, 12)

assert (
    era5["date"].min()
    == pd.Timestamp("2025-01-05")
)

assert (
    era5["date"].max()
    == pd.Timestamp("2026-05-19")
)

assert era5["date"].nunique() == 500
assert unique_grid_locations == 1361
assert duplicate_rows == 0
assert missing_values == 0

print("ERA5 validation passed.")

Dataset shape: (680500, 12)
First date: 2025-01-05 00:00:00
Last date: 2026-05-19 00:00:00
Unique dates: 500
Unique grid locations: 1361
Duplicate rows: 0
Missing values: 0
ERA5 validation passed.


In [93]:
ERA5_RERUN_PATH = (
    PROJECT_FOLDER
    / (
        "era5_nepal_2025_01_05_"
        "to_2026_05_19_full_rerun_v2.parquet"
    )
)

era5.to_parquet(
    ERA5_RERUN_PATH,
    index=False
)

print(
    "Saved full-rerun ERA5 dataset:",
    ERA5_RERUN_PATH
)

Saved full-rerun ERA5 dataset: /content/drive/MyDrive/Wildfire_ML_Project/era5_nepal_2025_01_05_to_2026_05_19_full_rerun_v2.parquet


In [94]:
FIRMS_FOLDER = (
    PROJECT_FOLDER
    / "FIRMS"
)

firms_files = sorted(
    FIRMS_FOLDER.glob("*.csv")
)

assert firms_files, (
    "No FIRMS CSV files were found."
)

firms_frames = []
firms_source_summary = []

for file_path in firms_files:
    current_df = pd.read_csv(
        file_path
    )

    current_df["acq_date"] = pd.to_datetime(
        current_df["acq_date"],
        errors="coerce"
    ).dt.normalize()

    firms_source_summary.append({
        "file": file_path.name,
        "rows": len(current_df),
        "first_date": (
            current_df["acq_date"].min()
        ),
        "last_date": (
            current_df["acq_date"].max()
        )
    })

    firms_frames.append(
        current_df
    )

firms_source_summary = pd.DataFrame(
    firms_source_summary
)

display(
    firms_source_summary
)

firms = pd.concat(
    firms_frames,
    ignore_index=True
)

print(
    "Combined raw FIRMS shape:",
    firms.shape
)

,file,rows,first_date,last_date
0,fire_archive_M-C61_751953.csv,3756,2025-01-05,2026-01-05
1,fire_archive_M-C61_751954.csv,464,2026-01-06,2026-02-28
2,fire_nrt_M-C61_751954.csv,2399,2026-03-01,2026-05-18


Combined raw FIRMS shape: (6619, 15)


In [95]:
DATA_START_DATE = pd.Timestamp(
    "2025-01-05"
)

DATA_END_DATE_EXCLUSIVE = pd.Timestamp(
    "2026-05-20"
)

firms = firms.loc[
    (
        firms["acq_date"]
        >= DATA_START_DATE
    )
    &
    (
        firms["acq_date"]
        < DATA_END_DATE_EXCLUSIVE
    )
].copy()

In [96]:
firms = (
    firms
    .dropna(
        subset=[
            "acq_date",
            "latitude",
            "longitude",
            "brightness",
            "confidence",
            "frp"
        ]
    )
    .drop_duplicates(
        subset=[
            "latitude",
            "longitude",
            "acq_date",
            "acq_time",
            "satellite",
            "instrument",
            "brightness",
            "frp"
        ]
    )
    .copy()
)

In [97]:
firms = firms.loc[
    firms["latitude"].between(
        26.0,
        31.0
    )
    &
    firms["longitude"].between(
        79.5,
        89.0
    )
].copy()

firms = (
    firms
    .sort_values(
        ["acq_date", "latitude", "longitude"]
    )
    .reset_index(drop=True)
)

In [98]:
post_march_firms = int(
    (
        firms["acq_date"]
        >= pd.Timestamp("2026-03-01")
    ).sum()
)

print("Cleaned FIRMS shape:", firms.shape)

print(
    "FIRMS period:",
    firms["acq_date"].min(),
    "to",
    firms["acq_date"].max()
)

print(
    "Detections from March onward:",
    post_march_firms
)

assert post_march_firms > 0, (
    "No FIRMS records exist after February 2026."
)

Cleaned FIRMS shape: (6618, 15)
FIRMS period: 2025-01-05 00:00:00 to 2026-05-18 00:00:00
Detections from March onward: 2399


In [99]:
monthly_firms_check = (
    firms
    .assign(
        month=(
            firms["acq_date"]
            .dt.to_period("M")
            .astype(str)
        )
    )
    .groupby(
        "month",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "firms_detections"
        }
    )
)

display(
    monthly_firms_check.tail(8)
)

,month,firms_detections
8,2025-10,17
9,2025-11,125
10,2025-12,148
11,2026-01,229
12,2026-02,242
13,2026-03,335
14,2026-04,2027
15,2026-05,37


In [101]:
# Standardise FIRMS text metadata columns before Parquet export.
text_columns = [
    "satellite",
    "instrument",
    "version",
    "daynight"
]

for column in text_columns:
    if column in firms.columns:
        firms[column] = (
            firms[column]
            .astype("string")
        )

print(
    firms[text_columns].dtypes
)

satellite     string[python]
instrument    string[python]
version       string[python]
daynight      string[python]
dtype: object


In [102]:
FIRMS_RERUN_PATH = (
    PROJECT_FOLDER
    / (
        "firms_nepal_cleaned_"
        "2025_01_05_to_2026_05_19_"
        "full_rerun_v2.parquet"
    )
)

firms.to_parquet(
    FIRMS_RERUN_PATH,
    index=False
)

print(
    "Saved full-rerun FIRMS dataset:",
    FIRMS_RERUN_PATH
)

Saved full-rerun FIRMS dataset: /content/drive/MyDrive/Wildfire_ML_Project/firms_nepal_cleaned_2025_01_05_to_2026_05_19_full_rerun_v2.parquet


In [103]:
ERA5_PATH = ERA5_RERUN_PATH
FIRMS_PATH = FIRMS_RERUN_PATH

era5 = pd.read_parquet(
    ERA5_PATH
)

firms = pd.read_parquet(
    FIRMS_PATH
)

era5["date"] = pd.to_datetime(
    era5["date"]
).dt.normalize()

firms["acq_date"] = pd.to_datetime(
    firms["acq_date"]
).dt.normalize()

In [106]:
print(firms.columns.tolist())

['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']


In [107]:
import numpy as np
import pandas as pd

from sklearn.neighbors import BallTree


era5_grid = (
    era5[
        ["latitude", "longitude"]
    ]
    .drop_duplicates()
    .sort_values(
        ["latitude", "longitude"]
    )
    .reset_index(drop=True)
)

print(
    "Unique ERA5 grid cells:",
    len(era5_grid)
)

assert len(era5_grid) == 1361

Unique ERA5 grid cells: 1361


In [108]:
required_era5_columns = {
    "latitude",
    "longitude"
}

required_firms_columns = {
    "latitude",
    "longitude"
}

assert required_era5_columns.issubset(
    era5.columns
), "ERA5 coordinate columns are missing."

assert required_firms_columns.issubset(
    firms.columns
), "FIRMS coordinate columns are missing."

print("Coordinate columns validated.")

Coordinate columns validated.


In [109]:
era5_coordinates_rad = np.radians(
    era5_grid[
        ["latitude", "longitude"]
    ].to_numpy(
        dtype=float
    )
)

firms_coordinates_rad = np.radians(
    firms[
        ["latitude", "longitude"]
    ].to_numpy(
        dtype=float
    )
)

print(
    "ERA5 coordinate shape:",
    era5_coordinates_rad.shape
)

print(
    "FIRMS coordinate shape:",
    firms_coordinates_rad.shape
)

ERA5 coordinate shape: (1361, 2)
FIRMS coordinate shape: (6618, 2)


In [110]:
grid_tree = BallTree(
    era5_coordinates_rad,
    metric="haversine"
)

distances_rad, nearest_indices = (
    grid_tree.query(
        firms_coordinates_rad,
        k=1
    )
)

print(
    "Matching completed for",
    len(firms),
    "FIRMS records."
)

Matching completed for 6618 FIRMS records.


In [111]:
EARTH_RADIUS_KM = 6371.0088

firms = firms.copy()

firms["match_distance_km"] = (
    distances_rad[:, 0]
    * EARTH_RADIUS_KM
)

firms["grid_latitude"] = (
    era5_grid.iloc[
        nearest_indices[:, 0]
    ]["latitude"]
    .to_numpy()
)

firms["grid_longitude"] = (
    era5_grid.iloc[
        nearest_indices[:, 0]
    ]["longitude"]
    .to_numpy()
)

In [112]:
matching_columns = [
    "match_distance_km",
    "grid_latitude",
    "grid_longitude"
]

print(
    firms[
        matching_columns
    ].head()
)

print(
    firms[
        "match_distance_km"
    ].describe()
)

   match_distance_km  grid_latitude  grid_longitude
0          20.601684      29.899725       80.601192
1          24.041537      28.299718       81.301195
2          16.028894      28.299718       81.301195
3          15.224460      28.299718       81.301195
4          15.420512      28.299718       81.301195
count    6618.000000
mean        6.247461
std         4.691654
min         0.147388
25%         3.428607
50%         4.833649
75%         6.548908
max        26.451458
Name: match_distance_km, dtype: float64


In [113]:
MAX_MATCH_DISTANCE_KM = 15.0

firms_matched = firms.loc[
    firms["match_distance_km"]
    <= MAX_MATCH_DISTANCE_KM
].copy()

print(
    "FIRMS records before filtering:",
    len(firms)
)

print(
    "Records retained:",
    len(firms_matched)
)

print(
    "Records excluded:",
    len(firms) - len(firms_matched)
)

FIRMS records before filtering: 6618
Records retained: 6013
Records excluded: 605


In [114]:
matched_post_march = int(
    (
        firms_matched["acq_date"]
        >= pd.Timestamp("2026-03-01")
    ).sum()
)

print(
    "Matched post-March detections:",
    matched_post_march
)

assert matched_post_march > 0, (
    "No post-March FIRMS detections survived "
    "the spatial matching process."
)

Matched post-March detections: 2188


## Daily FIRMS Fire-Label Construction

Multiple FIRMS detections may occur in the same ERA5 grid cell on the same
date. These detections are aggregated into one grid-cell daily record before
being merged with the ERA5 weather dataset.

In [116]:
fire_daily = (
    firms_matched
    .groupby(
        [
            "acq_date",
            "grid_latitude",
            "grid_longitude"
        ],
        as_index=False
    )
    .agg(
        fire_count=(
            "frp",
            "size"
        ),
        frp_sum=(
            "frp",
            "sum"
        ),
        frp_max=(
            "frp",
            "max"
        ),
        confidence_mean=(
            "confidence",
            "mean"
        ),
        brightness_mean=(
            "brightness",
            "mean"
        ),
        match_distance_mean_km=(
            "match_distance_km",
            "mean"
        ),
        match_distance_max_km=(
            "match_distance_km",
            "max"
        )
    )
)

In [117]:
fire_daily = fire_daily.rename(
    columns={
        "acq_date": "date",
        "grid_latitude": "latitude",
        "grid_longitude": "longitude"
    }
)

fire_daily["date"] = pd.to_datetime(
    fire_daily["date"],
    errors="coerce"
).dt.normalize()

fire_daily["fire_detected"] = 1
fire_daily["fire_detected"] = (
    fire_daily["fire_detected"]
    .astype("int8")
)

print(
    "fire_daily columns:"
)

print(
    fire_daily.columns.tolist()
)

fire_daily columns:
['date', 'latitude', 'longitude', 'fire_count', 'frp_sum', 'frp_max', 'confidence_mean', 'brightness_mean', 'match_distance_mean_km', 'match_distance_max_km', 'fire_detected']


In [118]:
print(
    "FIRMS detections retained after matching:",
    len(firms_matched)
)

print(
    "Unique fire grid-cell days:",
    len(fire_daily)
)

print(
    "First fire-label date:",
    fire_daily["date"].min()
)

print(
    "Last fire-label date:",
    fire_daily["date"].max()
)

print(
    "Unique fire-label dates:",
    fire_daily["date"].nunique()
)

print(
    "Unique fire-affected grid cells:",
    fire_daily[
        ["latitude", "longitude"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicate fire grid-cell days:",
    fire_daily.duplicated(
        [
            "date",
            "latitude",
            "longitude"
        ]
    ).sum()
)

FIRMS detections retained after matching: 6013
Unique fire grid-cell days: 3461
First fire-label date: 2025-01-06 00:00:00
Last fire-label date: 2026-05-18 00:00:00
Unique fire-label dates: 293
Unique fire-affected grid cells: 788
Duplicate fire grid-cell days: 0


In [119]:
post_march_fire_grid_days = int(
    (
        fire_daily["date"]
        >= pd.Timestamp("2026-03-01")
    ).sum()
)

print(
    "Post-March fire grid-cell days:",
    post_march_fire_grid_days
)

assert post_march_fire_grid_days > 0, (
    "No fire grid-cell days exist after February 2026."
)

assert fire_daily.duplicated(
    [
        "date",
        "latitude",
        "longitude"
    ]
).sum() == 0

print(
    "Daily fire-label validation passed."
)

Post-March fire grid-cell days: 1201
Daily fire-label validation passed.


In [120]:
monthly_fire_labels = (
    fire_daily
    .assign(
        month=(
            fire_daily["date"]
            .dt.to_period("M")
            .astype(str)
        )
    )
    .groupby(
        "month",
        as_index=False
    )
    .agg(
        fire_grid_cell_days=(
            "fire_detected",
            "sum"
        ),
        satellite_detections=(
            "fire_count",
            "sum"
        ),
        total_frp=(
            "frp_sum",
            "sum"
        )
    )
)

monthly_fire_labels.tail(8)

,month,fire_grid_cell_days,satellite_detections,total_frp
8,2025-10,2,2,16.00
9,2025-11,76,99,1058.70
10,2025-12,103,134,2114.00
11,2026-01,135,198,5050.40
12,2026-02,131,183,4643.50
13,2026-03,194,288,4795.87
14,2026-04,984,1866,29426.43
15,2026-05,23,34,449.44


In [121]:
era5["date"] = pd.to_datetime(
    era5["date"],
    errors="coerce"
).dt.normalize()

era5["latitude"] = pd.to_numeric(
    era5["latitude"],
    errors="coerce"
)

era5["longitude"] = pd.to_numeric(
    era5["longitude"],
    errors="coerce"
)

fire_daily["latitude"] = pd.to_numeric(
    fire_daily["latitude"],
    errors="coerce"
)

fire_daily["longitude"] = pd.to_numeric(
    fire_daily["longitude"],
    errors="coerce"
)

print(
    "ERA5 date type:",
    era5["date"].dtype
)

print(
    "Fire-label date type:",
    fire_daily["date"].dtype
)

ERA5 date type: datetime64[ns]
Fire-label date type: datetime64[ns]


## ERA5 and FIRMS Dataset Merge

The aggregated FIRMS fire labels are left-joined with the complete ERA5
grid-cell daily dataset.

Every ERA5 grid-cell date is retained. Rows with a matching FIRMS record are
labelled as fire detections, while unmatched rows are labelled as non-fire
observations.

In [122]:
wildfire_base = era5.merge(
    fire_daily,
    on=[
        "date",
        "latitude",
        "longitude"
    ],
    how="left",
    validate="one_to_one"
)

print(
    "ERA5 rows before merge:",
    len(era5)
)

print(
    "Rows after merge:",
    len(wildfire_base)
)

assert len(wildfire_base) == len(era5), (
    "The merge unexpectedly changed the ERA5 row count."
)

print(
    "ERA5-FIRMS merge completed."
)

ERA5 rows before merge: 680500
Rows after merge: 680500
ERA5-FIRMS merge completed.


In [123]:
wildfire_base["fire_detected"] = (
    wildfire_base["fire_detected"]
    .fillna(0)
    .astype("int8")
)

In [124]:
fire_metadata_columns = [
    "fire_count",
    "frp_sum",
    "frp_max",
    "confidence_mean",
    "brightness_mean",
    "match_distance_mean_km",
    "match_distance_max_km"
]

wildfire_base[
    fire_metadata_columns
] = (
    wildfire_base[
        fire_metadata_columns
    ]
    .fillna(0)
)

In [125]:
wildfire_base = (
    wildfire_base
    .sort_values(
        [
            "date",
            "latitude",
            "longitude"
        ]
    )
    .reset_index(drop=True)
)

In [126]:
unique_grid_locations = (
    wildfire_base[
        ["latitude", "longitude"]
    ]
    .drop_duplicates()
    .shape[0]
)

duplicate_rows = wildfire_base.duplicated(
    [
        "date",
        "latitude",
        "longitude"
    ]
).sum()

missing_values = (
    wildfire_base
    .isna()
    .sum()
    .sum()
)

total_positive_labels = int(
    wildfire_base[
        "fire_detected"
    ].sum()
)

post_march_positive_labels = int(
    wildfire_base.loc[
        wildfire_base["date"]
        >= pd.Timestamp("2026-03-01"),
        "fire_detected"
    ].sum()
)

print(
    "Final dataset shape:",
    wildfire_base.shape
)

print(
    "Date range:",
    wildfire_base["date"].min(),
    "to",
    wildfire_base["date"].max()
)

print(
    "Unique dates:",
    wildfire_base["date"].nunique()
)

print(
    "Unique grid locations:",
    unique_grid_locations
)

print(
    "Duplicate date-location rows:",
    duplicate_rows
)

print(
    "Missing values:",
    missing_values
)

print(
    "Total positive labels:",
    total_positive_labels
)

print(
    "Positive labels from March onward:",
    post_march_positive_labels
)

Final dataset shape: (680500, 20)
Date range: 2025-01-05 00:00:00 to 2026-05-19 00:00:00
Unique dates: 500
Unique grid locations: 1361
Duplicate date-location rows: 0
Missing values: 0
Total positive labels: 3461
Positive labels from March onward: 1201


In [127]:
assert len(wildfire_base) == len(era5)

assert wildfire_base["date"].nunique() == 500

assert unique_grid_locations == 1361

assert duplicate_rows == 0

assert missing_values == 0

assert (
    wildfire_base["date"].min()
    == pd.Timestamp("2025-01-05")
)

assert (
    wildfire_base["date"].max()
    == pd.Timestamp("2026-05-19")
)

assert post_march_positive_labels > 0, (
    "The corrected dataset still has no positive "
    "labels after February 2026."
)

print(
    "Corrected merged dataset validation passed."
)

Corrected merged dataset validation passed.


In [128]:
monthly_merged_labels = (
    wildfire_base
    .assign(
        month=(
            wildfire_base["date"]
            .dt.to_period("M")
            .astype(str)
        )
    )
    .groupby(
        "month",
        as_index=False
    )
    .agg(
        positive_grid_cell_days=(
            "fire_detected",
            "sum"
        )
    )
)

monthly_merged_labels.tail(8)

,month,positive_grid_cell_days
9,2025-10,2
10,2025-11,76
11,2025-12,103
12,2026-01,135
13,2026-02,131
14,2026-03,194
15,2026-04,984
16,2026-05,23


In [129]:
target_check = (
    wildfire_base
    .sort_values(
        [
            "latitude",
            "longitude",
            "date"
        ]
    )
    .copy()
)

target_check["fire_next_day"] = (
    target_check
    .groupby(
        [
            "latitude",
            "longitude"
        ]
    )["fire_detected"]
    .shift(-1)
)

In [130]:
test_period_target = target_check.loc[
    target_check["date"].between(
        pd.Timestamp("2026-03-01"),
        pd.Timestamp("2026-05-18")
    )
].copy()

test_positive_examples = int(
    test_period_target[
        "fire_next_day"
    ].sum()
)

print(
    "Next-day positive examples "
    "in the intended test period:",
    test_positive_examples
)

assert test_positive_examples > 0, (
    "The intended test period still contains "
    "no positive next-day examples."
)

print(
    "Test-period target validation passed."
)

Next-day positive examples in the intended test period: 1184
Test-period target validation passed.


In [131]:
FINAL_RERUN_PATH = (
    PROJECT_FOLDER
    / (
        "wildfire_base_grid_daily_same_day_"
        "full_rerun_v2.parquet"
    )
)

wildfire_base.to_parquet(
    FINAL_RERUN_PATH,
    index=False,
    engine="pyarrow"
)

print(
    "Saved corrected merged dataset:",
    FINAL_RERUN_PATH
)

Saved corrected merged dataset: /content/drive/MyDrive/Wildfire_ML_Project/wildfire_base_grid_daily_same_day_full_rerun_v2.parquet


In [132]:
saved_check = pd.read_parquet(
    FINAL_RERUN_PATH,
    columns=[
        "date",
        "latitude",
        "longitude",
        "fire_detected"
    ]
)

saved_check["date"] = pd.to_datetime(
    saved_check["date"]
).dt.normalize()

print(
    "Saved rows:",
    len(saved_check)
)

print(
    "Saved date range:",
    saved_check["date"].min(),
    "to",
    saved_check["date"].max()
)

print(
    "Saved positive labels:",
    int(
        saved_check[
            "fire_detected"
        ].sum()
    )
)

print(
    "Saved post-March positive labels:",
    int(
        saved_check.loc[
            saved_check["date"]
            >= pd.Timestamp("2026-03-01"),
            "fire_detected"
        ].sum()
    )
)

Saved rows: 680500
Saved date range: 2025-01-05 00:00:00 to 2026-05-19 00:00:00
Saved positive labels: 3461
Saved post-March positive labels: 1201


In [133]:
assert len(saved_check) == 680_500

assert saved_check["date"].nunique() == 500

assert (
    saved_check.loc[
        saved_check["date"]
        >= pd.Timestamp("2026-03-01"),
        "fire_detected"
    ].sum()
    > 0
)

print(
    "Saved corrected Parquet validation passed."
)

Saved corrected Parquet validation passed.


In [134]:
import shutil

CANONICAL_BASE_PATH = (
    PROJECT_FOLDER
    / "wildfire_base_grid_daily_same_day.parquet"
)

OLD_BASE_BACKUP_PATH = (
    PROJECT_FOLDER
    / (
        "wildfire_base_grid_daily_same_day_"
        "old_incomplete_backup.parquet"
    )
)

if (
    CANONICAL_BASE_PATH.exists()
    and not OLD_BASE_BACKUP_PATH.exists()
):
    shutil.copy2(
        CANONICAL_BASE_PATH,
        OLD_BASE_BACKUP_PATH
    )

    print(
        "Old dataset backed up:",
        OLD_BASE_BACKUP_PATH
    )
else:
    print(
        "Backup already exists or the old file "
        "was not found."
    )

Old dataset backed up: /content/drive/MyDrive/Wildfire_ML_Project/wildfire_base_grid_daily_same_day_old_incomplete_backup.parquet


In [135]:
shutil.copy2(
    FINAL_RERUN_PATH,
    CANONICAL_BASE_PATH
)

print(
    "Canonical dataset replaced:",
    CANONICAL_BASE_PATH
)

Canonical dataset replaced: /content/drive/MyDrive/Wildfire_ML_Project/wildfire_base_grid_daily_same_day.parquet


In [136]:
canonical_check = pd.read_parquet(
    CANONICAL_BASE_PATH,
    columns=[
        "date",
        "fire_detected"
    ]
)

canonical_check["date"] = pd.to_datetime(
    canonical_check["date"]
).dt.normalize()

print(
    "Canonical rows:",
    len(canonical_check)
)

print(
    "Canonical positives:",
    int(
        canonical_check[
            "fire_detected"
        ].sum()
    )
)

print(
    "Canonical post-March positives:",
    int(
        canonical_check.loc[
            canonical_check["date"]
            >= pd.Timestamp("2026-03-01"),
            "fire_detected"
        ].sum()
    )
)

Canonical rows: 680500
Canonical positives: 3461
Canonical post-March positives: 1201


In [137]:
NEW_PARQUET_PATH = (
    PROJECT_FOLDER
    / "wildfire_base_grid_daily_same_day_full_rerun_v2.parquet"
)

new_wildfire_data = pd.read_parquet(
    NEW_PARQUET_PATH
)

new_wildfire_data.head(10)

,date,latitude,longitude,temperature_mean_c,temperature_max_c,dewpoint_c,relative_humidity,u_wind_mps,v_wind_mps,wind_speed_mps,precipitation_mm,soil_moisture,fire_count,frp_sum,frp_max,confidence_mean,brightness_mean,match_distance_mean_km,match_distance_max_km,fire_detected
0,2025-01-05,26.399709,87.301223,18.407688,24.489526,12.324414,67.685346,-0.331734,-0.726165,0.798350,0.0,0.143019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2025-01-05,26.399709,87.601224,18.329319,24.785562,11.368685,63.864683,-0.588997,-0.787729,0.983582,0.0,0.142688,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2025-01-05,26.399709,88.001226,18.132461,24.924234,10.956413,62.915354,-0.752470,-0.670989,1.008185,0.0,0.127279,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2025-01-05,26.499709,86.601220,17.900039,23.536401,13.511914,75.519267,0.586336,-0.927133,1.096980,0.0,0.143495,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,2025-01-05,26.499709,86.701220,17.930231,23.616479,13.372103,74.692865,0.550448,-0.918792,1.071061,0.0,0.145238,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5,2025-01-05,26.499709,86.801221,17.941461,23.753198,13.347119,74.518718,0.460624,-0.839812,0.957841,0.0,0.141953,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
6,2025-01-05,26.499709,86.901221,18.046604,23.909448,13.362582,74.102405,0.358879,-0.738758,0.821315,0.0,0.143157,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
7,2025-01-05,26.499709,87.101222,18.126764,24.051187,12.743115,70.806577,0.057528,-0.684966,0.687377,0.0,0.138837,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
8,2025-01-05,26.499709,87.201222,18.135716,24.207437,12.349805,68.964976,-0.201749,-0.730783,0.758120,0.0,0.141702,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
9,2025-01-05,26.499709,87.301223,18.168268,24.392984,11.950147,67.038440,-0.437528,-0.781625,0.895751,0.0,0.141589,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
PERIOD_1_TABLE = """
`wildfire-project-502905.
era5_nepal_daily_2025_01_05_to_2026_01_05_1784440678767.
era5_nepal_daily_2025_01_05_to_2026_01_05`
""".replace("\n", "")

PERIOD_2_TABLE = """
`wildfire-project-502905.
era5_nepal_daily_2025_01_05_to_2026_01_05_1784440678767.
era5_nepal_daily_2026_01_06_to_2026_05_19`
""".replace("\n", "")

In [ ]:
print("Period 1:", PERIOD_1_TABLE)
print("Period 2:", PERIOD_2_TABLE)

Period 1: `wildfire-project-502905.era5_nepal_daily_2025_01_05_to_2026_01_05_1784440678767.era5_nepal_daily_2025_01_05_to_2026_01_05`
Period 2: `wildfire-project-502905.era5_nepal_daily_2025_01_05_to_2026_01_05_1784440678767.era5_nepal_daily_2026_01_06_to_2026_05_19`


In [ ]:
count_query = f"""
SELECT
    'period_1' AS period,
    COUNT(*) AS total_rows,
    MIN(SAFE_CAST(date AS DATE)) AS first_date,
    MAX(SAFE_CAST(date AS DATE)) AS last_date
FROM {PERIOD_1_TABLE}

UNION ALL

SELECT
    'period_2' AS period,
    COUNT(*) AS total_rows,
    MIN(SAFE_CAST(date AS DATE)) AS first_date,
    MAX(SAFE_CAST(date AS DATE)) AS last_date
FROM {PERIOD_2_TABLE}
"""

count_result = client.query(count_query).to_dataframe()

count_result

,period,total_rows,first_date,last_date
0,period_1,498126,2025-01-05,2026-01-05
1,period_2,73494,2026-01-06,2026-02-28


In [ ]:
preview_query = f"""
SELECT
    SAFE_CAST(date AS DATE) AS date,
    SAFE_CAST(latitude AS FLOAT64) AS latitude,
    SAFE_CAST(longitude AS FLOAT64) AS longitude,
    SAFE_CAST(temperature_mean_c AS FLOAT64) AS temperature_mean_c,
    SAFE_CAST(temperature_max_c AS FLOAT64) AS temperature_max_c,
    SAFE_CAST(dewpoint_c AS FLOAT64) AS dewpoint_c,
    SAFE_CAST(relative_humidity AS FLOAT64) AS relative_humidity,
    SAFE_CAST(u_wind_mps AS FLOAT64) AS u_wind_mps,
    SAFE_CAST(v_wind_mps AS FLOAT64) AS v_wind_mps,
    SAFE_CAST(wind_speed_mps AS FLOAT64) AS wind_speed_mps,
    SAFE_CAST(precipitation_mm AS FLOAT64) AS precipitation_mm,
    SAFE_CAST(soil_moisture AS FLOAT64) AS soil_moisture
FROM {PERIOD_1_TABLE}

UNION ALL

SELECT
    SAFE_CAST(date AS DATE),
    SAFE_CAST(latitude AS FLOAT64),
    SAFE_CAST(longitude AS FLOAT64),
    SAFE_CAST(temperature_mean_c AS FLOAT64),
    SAFE_CAST(temperature_max_c AS FLOAT64),
    SAFE_CAST(dewpoint_c AS FLOAT64),
    SAFE_CAST(relative_humidity AS FLOAT64),
    SAFE_CAST(u_wind_mps AS FLOAT64),
    SAFE_CAST(v_wind_mps AS FLOAT64),
    SAFE_CAST(wind_speed_mps AS FLOAT64),
    SAFE_CAST(precipitation_mm AS FLOAT64),
    SAFE_CAST(soil_moisture AS FLOAT64)
FROM {PERIOD_2_TABLE}

ORDER BY date, latitude, longitude
LIMIT 10
"""

preview_df = client.query(preview_query).to_dataframe(
    create_bqstorage_client=False
)

preview_df

,date,latitude,longitude,temperature_mean_c,temperature_max_c,dewpoint_c,relative_humidity,u_wind_mps,v_wind_mps,wind_speed_mps,precipitation_mm,soil_moisture
0,2025-01-05,26.399709,87.301223,18.407688,24.489526,12.324414,67.685346,-0.331734,-0.726165,0.798350,0.0,0.143019
1,2025-01-05,26.399709,87.601224,18.329319,24.785562,11.368685,63.864683,-0.588997,-0.787729,0.983582,0.0,0.142688
2,2025-01-05,26.399709,88.001226,18.132461,24.924234,10.956413,62.915354,-0.752470,-0.670989,1.008185,0.0,0.127279
3,2025-01-05,26.499709,86.601220,17.900039,23.536401,13.511914,75.519267,0.586336,-0.927133,1.096980,0.0,0.143495
4,2025-01-05,26.499709,86.701220,17.930231,23.616479,13.372103,74.692865,0.550448,-0.918792,1.071061,0.0,0.145238
5,2025-01-05,26.499709,86.801221,17.941461,23.753198,13.347119,74.518718,0.460624,-0.839812,0.957841,0.0,0.141953
6,2025-01-05,26.499709,86.901221,18.046604,23.909448,13.362582,74.102405,0.358879,-0.738758,0.821315,0.0,0.143157
7,2025-01-05,26.499709,87.101222,18.126764,24.051187,12.743115,70.806577,0.057528,-0.684966,0.687377,0.0,0.138837
8,2025-01-05,26.499709,87.201222,18.135716,24.207437,12.349805,68.964976,-0.201749,-0.730783,0.758120,0.0,0.141702
9,2025-01-05,26.499709,87.301223,18.168268,24.392984,11.950147,67.038440,-0.437528,-0.781625,0.895751,0.0,0.141589


In [ ]:
print("Shape:", preview_df.shape)
print("\nColumns:")
print(preview_df.columns.tolist())

print("\nData types:")
print(preview_df.dtypes)

Shape: (10, 12)

Columns:
['date', 'latitude', 'longitude', 'temperature_mean_c', 'temperature_max_c', 'dewpoint_c', 'relative_humidity', 'u_wind_mps', 'v_wind_mps', 'wind_speed_mps', 'precipitation_mm', 'soil_moisture']

Data types:
date                   dbdate
latitude              float64
longitude             float64
temperature_mean_c    float64
temperature_max_c     float64
dewpoint_c            float64
relative_humidity     float64
u_wind_mps            float64
v_wind_mps            float64
wind_speed_mps        float64
precipitation_mm      float64
soil_moisture         float64
dtype: object


In [ ]:
combined_query = f"""
SELECT
    SAFE_CAST(date AS DATE) AS date,
    SAFE_CAST(latitude AS FLOAT64) AS latitude,
    SAFE_CAST(longitude AS FLOAT64) AS longitude,
    SAFE_CAST(temperature_mean_c AS FLOAT64) AS temperature_mean_c,
    SAFE_CAST(temperature_max_c AS FLOAT64) AS temperature_max_c,
    SAFE_CAST(dewpoint_c AS FLOAT64) AS dewpoint_c,
    SAFE_CAST(relative_humidity AS FLOAT64) AS relative_humidity,
    SAFE_CAST(u_wind_mps AS FLOAT64) AS u_wind_mps,
    SAFE_CAST(v_wind_mps AS FLOAT64) AS v_wind_mps,
    SAFE_CAST(wind_speed_mps AS FLOAT64) AS wind_speed_mps,
    SAFE_CAST(precipitation_mm AS FLOAT64) AS precipitation_mm,
    SAFE_CAST(soil_moisture AS FLOAT64) AS soil_moisture
FROM {PERIOD_1_TABLE}

UNION ALL

SELECT
    SAFE_CAST(date AS DATE),
    SAFE_CAST(latitude AS FLOAT64),
    SAFE_CAST(longitude AS FLOAT64),
    SAFE_CAST(temperature_mean_c AS FLOAT64),
    SAFE_CAST(temperature_max_c AS FLOAT64),
    SAFE_CAST(dewpoint_c AS FLOAT64),
    SAFE_CAST(relative_humidity AS FLOAT64),
    SAFE_CAST(u_wind_mps AS FLOAT64),
    SAFE_CAST(v_wind_mps AS FLOAT64),
    SAFE_CAST(wind_speed_mps AS FLOAT64),
    SAFE_CAST(precipitation_mm AS FLOAT64),
    SAFE_CAST(soil_moisture AS FLOAT64)
FROM {PERIOD_2_TABLE}

ORDER BY date, latitude, longitude
"""

era5 = client.query(combined_query).to_dataframe(
    create_bqstorage_client=False
)

print("ERA5 download completed.")
print("Shape:", era5.shape)

ERA5 download completed.
Shape: (571620, 12)


In [ ]:
import pandas as pd
import numpy as np

era5["date"] = pd.to_datetime(
    era5["date"],
    errors="coerce"
)

numeric_columns = [
    "latitude",
    "longitude",
    "temperature_mean_c",
    "temperature_max_c",
    "dewpoint_c",
    "relative_humidity",
    "u_wind_mps",
    "v_wind_mps",
    "wind_speed_mps",
    "precipitation_mm",
    "soil_moisture"
]

for column in numeric_columns:
    era5[column] = pd.to_numeric(
        era5[column],
        errors="coerce"
    )

era5 = era5.sort_values(
    ["date", "latitude", "longitude"]
).reset_index(drop=True)

era5.head()

,date,latitude,longitude,temperature_mean_c,temperature_max_c,dewpoint_c,relative_humidity,u_wind_mps,v_wind_mps,wind_speed_mps,precipitation_mm,soil_moisture
0,2025-01-05,26.399709,87.301223,18.407688,24.489526,12.324414,67.685346,-0.331734,-0.726165,0.798350,0.0,0.143019
1,2025-01-05,26.399709,87.601224,18.329319,24.785562,11.368685,63.864683,-0.588997,-0.787729,0.983582,0.0,0.142688
2,2025-01-05,26.399709,88.001226,18.132461,24.924234,10.956413,62.915354,-0.752470,-0.670989,1.008185,0.0,0.127279
3,2025-01-05,26.499709,86.601220,17.900039,23.536401,13.511914,75.519267,0.586336,-0.927133,1.096980,0.0,0.143495
4,2025-01-05,26.499709,86.701220,17.930231,23.616479,13.372103,74.692865,0.550448,-0.918792,1.071061,0.0,0.145238


In [ ]:
print("Dataset shape:", era5.shape)

print(
    "First date:",
    era5["date"].min()
)

print(
    "Last date:",
    era5["date"].max()
)

print(
    "Unique dates:",
    era5["date"].nunique()
)

print(
    "Unique grid locations:",
    era5[
        ["latitude", "longitude"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate date-location rows:",
    era5.duplicated(
        subset=["date", "latitude", "longitude"]
    ).sum()
)

print(
    "Total missing values:",
    era5.isna().sum().sum()
)

Dataset shape: (571620, 12)
First date: 2025-01-05 00:00:00
Last date: 2026-02-28 00:00:00
Unique dates: 420
Unique grid locations: 1361
Duplicate date-location rows: 0
Total missing values: 0


In [ ]:
rows_per_date = (
    era5.groupby("date")
    .size()
)

print(
    "Minimum grid rows on one date:",
    rows_per_date.min()
)

print(
    "Maximum grid rows on one date:",
    rows_per_date.max()
)

print(
    "Dates with unusual grid counts:",
    (
        rows_per_date[
            rows_per_date != rows_per_date.mode().iloc[0]
        ]
    ).shape[0]
)

Minimum grid rows on one date: 1361
Maximum grid rows on one date: 1361
Dates with unusual grid counts: 0


In [ ]:
range_summary = era5[
    [
        "temperature_mean_c",
        "temperature_max_c",
        "relative_humidity",
        "wind_speed_mps",
        "precipitation_mm",
        "soil_moisture"
    ]
].agg(["min", "mean", "max"]).T

range_summary

,min,mean,max
temperature_mean_c,-26.842959,12.128333,34.869330
temperature_max_c,-22.319571,16.765133,41.340036
relative_humidity,9.365642,71.836203,99.797110
wind_speed_mps,0.000680,0.594635,6.554824
precipitation_mm,0.000000,5.190221,216.945015
soil_moisture,0.027384,0.306687,0.710594


In [ ]:
print(
    "Humidity outside 0–100:",
    (
        (era5["relative_humidity"] < 0)
        |
        (era5["relative_humidity"] > 100)
    ).sum()
)

print(
    "Negative wind speeds:",
    (era5["wind_speed_mps"] < 0).sum()
)

print(
    "Negative precipitation:",
    (era5["precipitation_mm"] < 0).sum()
)

print(
    "Maximum temperature below mean temperature:",
    (
        era5["temperature_max_c"]
        <
        era5["temperature_mean_c"]
    ).sum()
)

print(
    "Soil moisture outside 0–1:",
    (
        (era5["soil_moisture"] < 0)
        |
        (era5["soil_moisture"] > 1)
    ).sum()
)

Humidity outside 0–100: 0
Negative wind speeds: 0
Negative precipitation: 0
Maximum temperature below mean temperature: 0
Soil moisture outside 0–1: 0


In [ ]:
PERIOD_1_TABLE = (
    "`wildfire-project-502905."
    "era5_nepal_daily_2025_01_05_to_2026_01_05_1784440678767."
    "era5_nepal_daily_2025_01_05_to_2026_01_05`"
)

PERIOD_2_TABLE = (
    "`wildfire-project-502905."
    "era5_nepal_daily_2025_01_05_to_2026_01_05_1784440678767."
    "era5_nepal_daily_period_2_corrected`"
)

print(PERIOD_1_TABLE)
print(PERIOD_2_TABLE)

`wildfire-project-502905.era5_nepal_daily_2025_01_05_to_2026_01_05_1784440678767.era5_nepal_daily_2025_01_05_to_2026_01_05`
`wildfire-project-502905.era5_nepal_daily_2025_01_05_to_2026_01_05_1784440678767.era5_nepal_daily_period_2_corrected`


In [ ]:
corrected_check_query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(SAFE_CAST(date AS DATE)) AS first_date,
    MAX(SAFE_CAST(date AS DATE)) AS last_date,
    COUNT(DISTINCT SAFE_CAST(date AS DATE)) AS unique_dates,
    COUNT(
        DISTINCT CONCAT(
            CAST(SAFE_CAST(latitude AS FLOAT64) AS STRING),
            '|',
            CAST(SAFE_CAST(longitude AS FLOAT64) AS STRING)
        )
    ) AS unique_grid_locations
FROM {PERIOD_2_TABLE}
"""

corrected_check = client.query(
    corrected_check_query
).to_dataframe()

corrected_check

,total_rows,first_date,last_date,unique_dates,unique_grid_locations
0,182374,2026-01-06,2026-05-19,134,1361


In [ ]:
period_check_query = f"""
SELECT
    'period_1' AS period,
    COUNT(*) AS total_rows,
    MIN(SAFE_CAST(date AS DATE)) AS first_date,
    MAX(SAFE_CAST(date AS DATE)) AS last_date,
    COUNT(DISTINCT SAFE_CAST(date AS DATE)) AS unique_dates
FROM {PERIOD_1_TABLE}

UNION ALL

SELECT
    'period_2_corrected' AS period,
    COUNT(*) AS total_rows,
    MIN(SAFE_CAST(date AS DATE)) AS first_date,
    MAX(SAFE_CAST(date AS DATE)) AS last_date,
    COUNT(DISTINCT SAFE_CAST(date AS DATE)) AS unique_dates
FROM {PERIOD_2_TABLE}
"""

period_check = client.query(
    period_check_query
).to_dataframe()

period_check

,period,total_rows,first_date,last_date,unique_dates
0,period_1,498126,2025-01-05,2026-01-05,366
1,period_2_corrected,182374,2026-01-06,2026-05-19,134


In [ ]:
combined_query = f"""
SELECT
    SAFE_CAST(date AS DATE) AS date,
    SAFE_CAST(latitude AS FLOAT64) AS latitude,
    SAFE_CAST(longitude AS FLOAT64) AS longitude,
    SAFE_CAST(temperature_mean_c AS FLOAT64) AS temperature_mean_c,
    SAFE_CAST(temperature_max_c AS FLOAT64) AS temperature_max_c,
    SAFE_CAST(dewpoint_c AS FLOAT64) AS dewpoint_c,
    SAFE_CAST(relative_humidity AS FLOAT64) AS relative_humidity,
    SAFE_CAST(u_wind_mps AS FLOAT64) AS u_wind_mps,
    SAFE_CAST(v_wind_mps AS FLOAT64) AS v_wind_mps,
    SAFE_CAST(wind_speed_mps AS FLOAT64) AS wind_speed_mps,
    SAFE_CAST(precipitation_mm AS FLOAT64) AS precipitation_mm,
    SAFE_CAST(soil_moisture AS FLOAT64) AS soil_moisture
FROM {PERIOD_1_TABLE}

UNION ALL

SELECT
    SAFE_CAST(date AS DATE) AS date,
    SAFE_CAST(latitude AS FLOAT64) AS latitude,
    SAFE_CAST(longitude AS FLOAT64) AS longitude,
    SAFE_CAST(temperature_mean_c AS FLOAT64) AS temperature_mean_c,
    SAFE_CAST(temperature_max_c AS FLOAT64) AS temperature_max_c,
    SAFE_CAST(dewpoint_c AS FLOAT64) AS dewpoint_c,
    SAFE_CAST(relative_humidity AS FLOAT64) AS relative_humidity,
    SAFE_CAST(u_wind_mps AS FLOAT64) AS u_wind_mps,
    SAFE_CAST(v_wind_mps AS FLOAT64) AS v_wind_mps,
    SAFE_CAST(wind_speed_mps AS FLOAT64) AS wind_speed_mps,
    SAFE_CAST(precipitation_mm AS FLOAT64) AS precipitation_mm,
    SAFE_CAST(soil_moisture AS FLOAT64) AS soil_moisture
FROM {PERIOD_2_TABLE}

ORDER BY date, latitude, longitude
"""

era5 = client.query(
    combined_query
).to_dataframe(
    create_bqstorage_client=False
)

print("Combined download complete.")
print("Shape:", era5.shape)

Combined download complete.
Shape: (680500, 12)


In [ ]:
import pandas as pd

era5["date"] = pd.to_datetime(
    era5["date"],
    errors="coerce"
)

numeric_columns = [
    "latitude",
    "longitude",
    "temperature_mean_c",
    "temperature_max_c",
    "dewpoint_c",
    "relative_humidity",
    "u_wind_mps",
    "v_wind_mps",
    "wind_speed_mps",
    "precipitation_mm",
    "soil_moisture"
]

for column in numeric_columns:
    era5[column] = pd.to_numeric(
        era5[column],
        errors="coerce"
    )

era5 = (
    era5
    .sort_values(["date", "latitude", "longitude"])
    .reset_index(drop=True)
)

In [ ]:
print("Dataset shape:", era5.shape)
print("First date:", era5["date"].min())
print("Last date:", era5["date"].max())
print("Unique dates:", era5["date"].nunique())

print(
    "Unique grid locations:",
    era5[["latitude", "longitude"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicate date-location rows:",
    era5.duplicated(
        subset=["date", "latitude", "longitude"]
    ).sum()
)

print(
    "Total missing values:",
    era5.isna().sum().sum()
)

Dataset shape: (680500, 12)
First date: 2025-01-05 00:00:00
Last date: 2026-05-19 00:00:00
Unique dates: 500
Unique grid locations: 1361
Duplicate date-location rows: 0
Total missing values: 0


In [ ]:
rows_per_date = era5.groupby("date").size()

print("Minimum rows per date:", rows_per_date.min())
print("Maximum rows per date:", rows_per_date.max())

unusual_dates = rows_per_date[
    rows_per_date != 1361
]

print("Dates with unexpected row counts:", len(unusual_dates))

unusual_dates.head()

Minimum rows per date: 1361
Maximum rows per date: 1361
Dates with unexpected row counts: 0


,0
date,


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/Wildfire_ML_Project"
)

FIRMS_FOLDER = PROJECT_FOLDER / "FIRMS"

PROJECT_FOLDER.mkdir(parents=True, exist_ok=True)
FIRMS_FOLDER.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_FOLDER)
print("FIRMS folder:", FIRMS_FOLDER)

Project folder: /content/drive/MyDrive/Wildfire_ML_Project
FIRMS folder: /content/drive/MyDrive/Wildfire_ML_Project/FIRMS


In [ ]:
ERA5_PATH = (
    PROJECT_FOLDER
    / "era5_nepal_2025_01_05_to_2026_05_19.parquet"
)

era5.to_parquet(
    ERA5_PATH,
    index=False
)

print("Saved:", ERA5_PATH)

Saved: /content/drive/MyDrive/Wildfire_ML_Project/era5_nepal_2025_01_05_to_2026_05_19.parquet


In [ ]:
import pandas as pd

era5_test = pd.read_parquet(ERA5_PATH)

print("Shape:", era5_test.shape)
print("First date:", era5_test["date"].min())
print("Last date:", era5_test["date"].max())

Shape: (680500, 12)
First date: 2025-01-05 00:00:00
Last date: 2026-05-19 00:00:00


In [ ]:
del era5_test

In [ ]:
firms_files = sorted(
    FIRMS_FOLDER.glob("*.csv")
)

print("FIRMS CSV files found:")

for file_path in firms_files:
    print(file_path.name)

FIRMS CSV files found:
fire_archive_M-C61_751953.csv
fire_archive_M-C61_751954.csv


In [ ]:
firms_frames = []

for file_path in firms_files:
    current_df = pd.read_csv(file_path)

    print(
        file_path.name,
        current_df.shape
    )

    firms_frames.append(current_df)

firms = pd.concat(
    firms_frames,
    ignore_index=True
)

print("Combined FIRMS shape:", firms.shape)

fire_archive_M-C61_751953.csv (3756, 15)
fire_archive_M-C61_751954.csv (464, 15)
Combined FIRMS shape: (4220, 15)


In [ ]:
print("Columns:")

print(firms.columns.tolist())

firms.head()

Columns:
['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,30.0850,80.6012,304.3,1.6,1.3,2025-01-05,451,Terra,MODIS,44,61.03,290.8,15.3,D,0
1,28.3518,84.5684,311.2,1.6,1.2,2025-01-06,353,Terra,MODIS,29,61.03,290.1,21.0,D,0
2,28.3322,84.5470,344.7,1.6,1.2,2025-01-06,353,Terra,MODIS,94,61.03,293.4,101.3,D,0
3,28.3309,84.5593,311.7,1.6,1.2,2025-01-06,353,Terra,MODIS,46,61.03,287.3,21.2,D,0
4,28.3456,84.5735,312.4,1.6,1.2,2025-01-06,850,Aqua,MODIS,66,61.03,293.2,21.3,D,0


In [ ]:
firms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4220 entries, 0 to 4219
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   latitude    4220 non-null   float64
 1   longitude   4220 non-null   float64
 2   brightness  4220 non-null   float64
 3   scan        4220 non-null   float64
 4   track       4220 non-null   float64
 5   acq_date    4220 non-null   object 
 6   acq_time    4220 non-null   int64  
 7   satellite   4220 non-null   object 
 8   instrument  4220 non-null   object 
 9   confidence  4220 non-null   int64  
 10  version     4220 non-null   float64
 11  bright_t31  4220 non-null   float64
 12  frp         4220 non-null   float64
 13  daynight    4220 non-null   object 
 14  type        4220 non-null   int64  
dtypes: float64(8), int64(3), object(4)
memory usage: 494.7+ KB


In [ ]:
firms["acq_date"] = pd.to_datetime(
    firms["acq_date"],
    errors="coerce"
)

numeric_firms_columns = [
    "latitude",
    "longitude",
    "brightness",
    "scan",
    "track",
    "acq_time",
    "confidence",
    "bright_t31",
    "frp",
    "type"
]

for column in numeric_firms_columns:
    firms[column] = pd.to_numeric(
        firms[column],
        errors="coerce"
    )

In [ ]:
essential_columns = [
    "acq_date",
    "latitude",
    "longitude",
    "confidence",
    "frp"
]

firms = firms.dropna(
    subset=essential_columns
).copy()

print("Shape after removing invalid rows:", firms.shape)

Shape after removing invalid rows: (4220, 15)


In [ ]:
START_DATE = pd.Timestamp("2025-01-05")
END_DATE = pd.Timestamp("2026-05-19")

firms = firms[
    firms["acq_date"].between(
        START_DATE,
        END_DATE
    )
].copy()

firms = firms.sort_values(
    ["acq_date", "latitude", "longitude"]
).reset_index(drop=True)

In [ ]:
print("FIRMS shape:", firms.shape)
print("First FIRMS date:", firms["acq_date"].min())
print("Last FIRMS date:", firms["acq_date"].max())
print("Unique fire dates:", firms["acq_date"].nunique())

FIRMS shape: (4220, 15)
First FIRMS date: 2025-01-05 00:00:00
Last FIRMS date: 2026-02-28 00:00:00
Unique fire dates: 248


In [ ]:
firms.groupby(
    firms["acq_date"].dt.to_period("M")
).size()

,0
acq_date,
2025-01,238
2025-02,356
2025-03,949
2025-04,1790
2025-05,105
2025-06,13
2025-07,6
2025-09,1
2025-10,17


In [ ]:
print(
    "Exact duplicate records:",
    firms.duplicated().sum()
)

print(
    "Missing values by column:"
)

print(
    firms.isna().sum()
)

Exact duplicate records: 0
Missing values by column:
latitude      0
longitude     0
brightness    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_t31    0
frp           0
daynight      0
type          0
dtype: int64


In [ ]:
print(
    "Latitude range:",
    firms["latitude"].min(),
    "to",
    firms["latitude"].max()
)

print(
    "Longitude range:",
    firms["longitude"].min(),
    "to",
    firms["longitude"].max()
)

Latitude range: 13.5843 to 30.1113
Longitude range: -8.4036 to 88.2934


In [ ]:
firms = firms[
    firms["latitude"].between(26.0, 31.0)
    &
    firms["longitude"].between(79.5, 89.0)
].copy()

print(
    "Shape after broad coordinate filtering:",
    firms.shape
)

Shape after broad coordinate filtering: (4219, 15)


In [ ]:
firms[
    [
        "brightness",
        "confidence",
        "bright_t31",
        "frp"
    ]
].describe()

,brightness,confidence,bright_t31,frp
count,4219.000000,4219.000000,4219.000000,4219.000000
mean,318.162479,61.515051,298.993553,22.002512
std,12.644114,18.446518,7.920848,41.606028
min,300.000000,0.000000,266.500000,2.500000
25%,309.300000,50.000000,293.700000,7.900000
50%,317.000000,62.000000,299.200000,11.900000
75%,323.550000,74.000000,305.000000,22.500000
max,437.200000,100.000000,325.500000,1464.400000


In [ ]:
print("Satellites:")
print(firms["satellite"].value_counts())

print("\nInstruments:")
print(firms["instrument"].value_counts())

print("\nDay/night:")
print(firms["daynight"].value_counts())

Satellites:
satellite
Aqua     3141
Terra    1078
Name: count, dtype: int64

Instruments:
instrument
MODIS    4219
Name: count, dtype: int64

Day/night:
daynight
D    3641
N     578
Name: count, dtype: int64


In [ ]:
firms["confidence"].describe()

,confidence
count,4219.000000
mean,61.515051
std,18.446518
min,0.000000
25%,50.000000
50%,62.000000
75%,74.000000
max,100.000000


In [ ]:
FIRMS_CLEAN_PATH = (
    PROJECT_FOLDER
    / "firms_nepal_cleaned_2025_01_05_to_2026_05_19.parquet"
)

firms.to_parquet(
    FIRMS_CLEAN_PATH,
    index=False
)

print("Saved cleaned FIRMS data:", FIRMS_CLEAN_PATH)

Saved cleaned FIRMS data: /content/drive/MyDrive/Wildfire_ML_Project/firms_nepal_cleaned_2025_01_05_to_2026_05_19.parquet


## 4. Geospatial Alignment and Fire-Label Construction

NASA FIRMS represents fire detections as geographic points, while ERA5-Land
represents weather observations as regularly spaced grid cells.

To create a supervised machine-learning dataset, every FIRMS detection is
assigned to its nearest ERA5-Land grid-cell centre. Fire detections occurring
within the same grid cell and on the same date are then aggregated.

The aggregated fire records are left-joined with the complete ERA5 grid-cell
daily dataset. ERA5 rows with a matching FIRMS detection are labelled as fire
observations, while unmatched grid-cell dates are labelled as non-fire
observations.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/Wildfire_ML_Project"
)

ERA5_PATH = (
    PROJECT_FOLDER
    / "era5_nepal_2025_01_05_to_2026_05_19.parquet"
)

FIRMS_PATH = (
    PROJECT_FOLDER
    / "firms_nepal_cleaned_2025_01_05_to_2026_05_19.parquet"
)

era5 = pd.read_parquet(ERA5_PATH)
firms = pd.read_parquet(FIRMS_PATH)

era5["date"] = pd.to_datetime(era5["date"])
firms["acq_date"] = pd.to_datetime(firms["acq_date"])

print("ERA5 shape:", era5.shape)
print("FIRMS shape:", firms.shape)

ERA5 shape: (680500, 12)
FIRMS shape: (4219, 15)


In [ ]:
print(
    "ERA5 period:",
    era5["date"].min().date(),
    "to",
    era5["date"].max().date()
)

print(
    "FIRMS period:",
    firms["acq_date"].min().date(),
    "to",
    firms["acq_date"].max().date()
)

ERA5 period: 2025-01-05 to 2026-05-19
FIRMS period: 2025-01-05 to 2026-02-28


In [ ]:
era5_grid = (
    era5[["latitude", "longitude"]]
    .drop_duplicates()
    .sort_values(["latitude", "longitude"])
    .reset_index(drop=True)
)

print("Unique ERA5 grid cells:", len(era5_grid))

Unique ERA5 grid cells: 1361


In [ ]:
from sklearn.neighbors import BallTree

# Convert latitude and longitude from degrees to radians
era5_coordinates_rad = np.radians(
    era5_grid[["latitude", "longitude"]].to_numpy()
)

firms_coordinates_rad = np.radians(
    firms[["latitude", "longitude"]].to_numpy()
)

# Build the spatial search tree
grid_tree = BallTree(
    era5_coordinates_rad,
    metric="haversine"
)

# Find the closest ERA5 grid centre for every FIRMS record
distances_rad, nearest_indices = grid_tree.query(
    firms_coordinates_rad,
    k=1
)

EARTH_RADIUS_KM = 6371.0088

firms["match_distance_km"] = (
    distances_rad[:, 0] * EARTH_RADIUS_KM
)

firms["grid_latitude"] = (
    era5_grid.iloc[
        nearest_indices[:, 0]
    ]["latitude"]
    .to_numpy()
)

firms["grid_longitude"] = (
    era5_grid.iloc[
        nearest_indices[:, 0]
    ]["longitude"]
    .to_numpy()
)

In [ ]:
match_summary = firms["match_distance_km"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
)

match_summary

,match_distance_km
count,4219.000000
mean,6.284292
std,4.737442
min,0.178388
50%,4.838191
90%,14.591059
95%,17.092041
99%,21.412332
max,26.451458


In [ ]:
print(
    "Maximum matching distance:",
    round(firms["match_distance_km"].max(), 2),
    "km"
)

print(
    "Records matched farther than 15 km:",
    (firms["match_distance_km"] > 15).sum()
)

print(
    "Records matched farther than 20 km:",
    (firms["match_distance_km"] > 20).sum()
)

Maximum matching distance: 26.45 km
Records matched farther than 15 km: 394
Records matched farther than 20 km: 92


In [ ]:
distant_matches = (
    firms.loc[
        firms["match_distance_km"] > 15,
        [
            "acq_date",
            "latitude",
            "longitude",
            "grid_latitude",
            "grid_longitude",
            "match_distance_km"
        ]
    ]
    .sort_values(
        "match_distance_km",
        ascending=False
    )
)

print(
    "Distant matches:",
    len(distant_matches)
)

distant_matches.head(10)

Distant matches: 394


,acq_date,latitude,longitude,grid_latitude,grid_longitude,match_distance_km
1161,2025-03-27,28.3557,80.7859,28.499718,81.001194,26.451458
4099,2026-02-16,28.3647,80.7716,28.599719,80.801193,26.292518
3922,2026-01-21,28.3666,80.7689,28.599719,80.801193,26.113117
516,2025-02-17,28.3437,80.8045,28.499718,81.001194,25.902954
1162,2025-03-27,28.3572,80.7958,28.499718,81.001194,25.583873
3923,2026-01-21,28.3764,80.7666,28.599719,80.801193,25.061055
418,2025-02-07,26.9813,84.4963,27.099712,84.701211,24.191446
1,2025-01-06,28.1655,81.1088,28.299718,81.301195,24.041537
1783,2025-04-01,28.3804,80.7967,28.499718,81.001194,23.996060
1784,2025-04-01,28.3852,80.7898,28.599719,80.801193,23.879419


In [ ]:
firms_matched = firms[
    firms["match_distance_km"] <= 15
].copy()

print(
    "Records retained:",
    len(firms_matched)
)

print(
    "Records excluded:",
    len(firms) - len(firms_matched)
)

Records retained: 3825
Records excluded: 394


In [ ]:
fire_daily = (
    firms_matched
    .groupby(
        [
            "acq_date",
            "grid_latitude",
            "grid_longitude"
        ]
    )
    .agg(
        fire_count=(
            "frp",
            "size"
        ),
        frp_sum=(
            "frp",
            "sum"
        ),
        frp_max=(
            "frp",
            "max"
        ),
        confidence_mean=(
            "confidence",
            "mean"
        ),
        brightness_mean=(
            "brightness",
            "mean"
        ),
        match_distance_mean_km=(
            "match_distance_km",
            "mean"
        ),
        match_distance_max_km=(
            "match_distance_km",
            "max"
        )
    )
    .reset_index()
)

In [ ]:
# Aggregate multiple FIRMS detections occurring in the same
# ERA5 grid cell on the same date.

fire_daily = (
    firms_matched
    .groupby(
        [
            "acq_date",
            "grid_latitude",
            "grid_longitude"
        ],
        as_index=False
    )
    .agg(
        fire_count=("frp", "size"),
        frp_sum=("frp", "sum"),
        frp_max=("frp", "max"),
        confidence_mean=("confidence", "mean"),
        brightness_mean=("brightness", "mean"),
        match_distance_mean_km=("match_distance_km", "mean"),
        match_distance_max_km=("match_distance_km", "max")
    )
)

# Rename the matching keys to exactly match the ERA5 dataset.
fire_daily = fire_daily.rename(
    columns={
        "acq_date": "date",
        "grid_latitude": "latitude",
        "grid_longitude": "longitude"
    }
)

# Standardise date type.
fire_daily["date"] = pd.to_datetime(
    fire_daily["date"],
    errors="coerce"
)

# Create the binary fire label.
fire_daily["fire_detected"] = 1

print("fire_daily columns:")
print(fire_daily.columns.tolist())

fire_daily columns:
['date', 'latitude', 'longitude', 'fire_count', 'frp_sum', 'frp_max', 'confidence_mean', 'brightness_mean', 'match_distance_mean_km', 'match_distance_max_km', 'fire_detected']


In [ ]:
required_columns = {
    "date",
    "latitude",
    "longitude",
    "fire_count",
    "frp_sum",
    "frp_max",
    "confidence_mean",
    "brightness_mean",
    "fire_detected"
}

missing_columns = required_columns.difference(
    fire_daily.columns
)

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

print(
    "Original FIRMS detections:",
    len(firms_matched)
)

print(
    "Unique fire grid-cell days:",
    len(fire_daily)
)

print(
    "Unique fire dates:",
    fire_daily["date"].nunique()
)

print(
    "Unique fire-affected grid cells:",
    fire_daily[
        ["latitude", "longitude"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate fire grid-cell days:",
    fire_daily.duplicated(
        ["date", "latitude", "longitude"]
    ).sum()
)

Original FIRMS detections: 3825
Unique fire grid-cell days: 2260
Unique fire dates: 232
Unique fire-affected grid cells: 710
Duplicate fire grid-cell days: 0


In [ ]:
era5["date"] = pd.to_datetime(
    era5["date"],
    errors="coerce"
)

era5["latitude"] = pd.to_numeric(
    era5["latitude"],
    errors="coerce"
)

era5["longitude"] = pd.to_numeric(
    era5["longitude"],
    errors="coerce"
)

fire_daily["latitude"] = pd.to_numeric(
    fire_daily["latitude"],
    errors="coerce"
)

fire_daily["longitude"] = pd.to_numeric(
    fire_daily["longitude"],
    errors="coerce"
)

print("ERA5 date type:", era5["date"].dtype)
print("FIRMS date type:", fire_daily["date"].dtype)

ERA5 date type: datetime64[ns]
FIRMS date type: datetime64[ns]


In [ ]:
wildfire_base = era5.merge(
    fire_daily,
    on=[
        "date",
        "latitude",
        "longitude"
    ],
    how="left",
    validate="one_to_one"
)

print("ERA5 rows before merge:", len(era5))
print("Rows after merge:", len(wildfire_base))

assert len(wildfire_base) == len(era5), (
    "The merge unexpectedly changed the ERA5 row count."
)

ERA5 rows before merge: 680500
Rows after merge: 680500


In [ ]:
wildfire_base["fire_detected"] = (
    wildfire_base["fire_detected"]
    .fillna(0)
    .astype("int8")
)

In [ ]:
fire_metadata_columns = [
    "fire_count",
    "frp_sum",
    "frp_max",
    "confidence_mean",
    "brightness_mean",
    "match_distance_mean_km",
    "match_distance_max_km"
]

wildfire_base[fire_metadata_columns] = (
    wildfire_base[fire_metadata_columns]
    .fillna(0)
)

In [ ]:
print("Dataset shape:", wildfire_base.shape)

print(
    "Date range:",
    wildfire_base["date"].min(),
    "to",
    wildfire_base["date"].max()
)

print(
    "Unique dates:",
    wildfire_base["date"].nunique()
)

print(
    "Unique grid locations:",
    wildfire_base[
        ["latitude", "longitude"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate date-location rows:",
    wildfire_base.duplicated(
        ["date", "latitude", "longitude"]
    ).sum()
)

print(
    "Total missing values:",
    wildfire_base.isna().sum().sum()
)

Dataset shape: (680500, 20)
Date range: 2025-01-05 00:00:00 to 2026-05-19 00:00:00
Unique dates: 500
Unique grid locations: 1361
Duplicate date-location rows: 0
Total missing values: 0


In [ ]:
class_counts = (
    wildfire_base["fire_detected"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

class_percentages = (
    class_counts
    .divide(len(wildfire_base))
    .multiply(100)
)

class_summary = pd.DataFrame({
    "class": [
        "No FIRMS hotspot",
        "FIRMS hotspot"
    ],
    "rows": class_counts.values,
    "percentage": class_percentages.values
})

class_summary

,class,rows,percentage
0,No FIRMS hotspot,678240,99.667891
1,FIRMS hotspot,2260,0.332109


In [ ]:
fire_rows = wildfire_base[
    wildfire_base["fire_detected"] == 1
].copy()

fire_rows["month"] = (
    fire_rows["date"]
    .dt.to_period("M")
    .astype(str)
)

monthly_fire_summary = (
    fire_rows
    .groupby("month", as_index=False)
    .agg(
        fire_grid_cell_days=(
            "fire_detected",
            "sum"
        ),
        satellite_detections=(
            "fire_count",
            "sum"
        ),
        total_frp=(
            "frp_sum",
            "sum"
        )
    )
)

monthly_fire_summary

,month,fire_grid_cell_days,satellite_detections,total_frp
0,2025-01,146,222.0,5903.1
1,2025-02,216,331.0,12907.3
2,2025-03,523,878.0,20037.6
3,2025-04,838,1663.0,31600.4
4,2025-05,73,96.0,1343.4
5,2025-06,11,13.0,233.3
6,2025-07,5,5.0,39.0
7,2025-09,1,1.0,7.1
8,2025-10,2,2.0,16.0
9,2025-11,76,99.0,1058.7


In [ ]:
BASE_DATASET_PATH = (
    PROJECT_FOLDER
    / "wildfire_base_grid_daily_same_day.parquet"
)

wildfire_base.to_parquet(
    BASE_DATASET_PATH,
    index=False
)

print(
    "Saved base dataset:",
    BASE_DATASET_PATH
)

Saved base dataset: /content/drive/MyDrive/Wildfire_ML_Project/wildfire_base_grid_daily_same_day.parquet


In [ ]:
base_test = pd.read_parquet(
    BASE_DATASET_PATH
)

print("Saved shape:", base_test.shape)
print(
    "Fire-labelled rows:",
    base_test["fire_detected"].sum()
)

Saved shape: (680500, 20)
Fire-labelled rows: 2260


In [ ]:
base_test.head()

,date,latitude,longitude,temperature_mean_c,temperature_max_c,dewpoint_c,relative_humidity,u_wind_mps,v_wind_mps,wind_speed_mps,precipitation_mm,soil_moisture,fire_count,frp_sum,frp_max,confidence_mean,brightness_mean,match_distance_mean_km,match_distance_max_km,fire_detected
0,2025-01-05,26.399709,87.301223,18.407688,24.489526,12.324414,67.685346,-0.331734,-0.726165,0.798350,0.0,0.143019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2025-01-05,26.399709,87.601224,18.329319,24.785562,11.368685,63.864683,-0.588997,-0.787729,0.983582,0.0,0.142688,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2025-01-05,26.399709,88.001226,18.132461,24.924234,10.956413,62.915354,-0.752470,-0.670989,1.008185,0.0,0.127279,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2025-01-05,26.499709,86.601220,17.900039,23.536401,13.511914,75.519267,0.586336,-0.927133,1.096980,0.0,0.143495,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,2025-01-05,26.499709,86.701220,17.930231,23.616479,13.372103,74.692865,0.550448,-0.918792,1.071061,0.0,0.145238,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
del base_test